In [0]:
%pip install sentence-transformers databricks-sdk

In [0]:
dbutils.library.restartPython()

In [0]:
import base64
from urllib.parse import urlparse
from datetime import datetime, date
import requests
import psycopg2
from sentence_transformers import SentenceTransformer
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
secret = w.secrets.get_secret(scope="database", key="lakebase-url")
connection_string = base64.b64decode(secret.value).decode("utf-8")
parsed = urlparse(connection_string)

model = SentenceTransformer("all-MiniLM-L6-v2")

def get_conn():
    return psycopg2.connect(
        host=parsed.hostname, port=parsed.port or 5432, dbname=parsed.path.lstrip("/"),
        user=parsed.username, password=parsed.password, sslmode="require",
    )

In [0]:
def search_destinations(query: str, top_k: int = 3) -> list[dict]:
    """Wrapper that calls the Unity Catalog Function via Spark SQL."""
    result = spark.sql(
        "SELECT * FROM trip_planner.silver.search_destinations(?, ?)",
        args=[query, top_k]
    )
    return [row.asDict() for row in result.collect()]

def get_weather(destination: str) -> dict:
    """Get current live weather and air quality for a destination."""
    geo_resp = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": destination, "count": 1}, timeout=10
    )
    geo_data = geo_resp.json()
    if "results" not in geo_data or not geo_data["results"]:
        return {"error": f"Could not find location: {destination}"}
    loc = geo_data["results"][0]
    lat, lon = loc["latitude"], loc["longitude"]
    weather_resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": "true"}, timeout=10
    )
    weather_data = weather_resp.json()["current_weather"]
    air_resp = requests.get(
        "https://air-quality-api.open-meteo.com/v1/air-quality",
        params={"latitude": lat, "longitude": lon, "current": "pm10,uv_index"}, timeout=10
    )
    air_data = air_resp.json()["current"]
    return {
        "destination": destination, "temperature_c": weather_data["temperature"],
        "windspeed": weather_data["windspeed"], "pm10": air_data.get("pm10"),
        "uv_index": air_data.get("uv_index"),
    }

def create_trip(user_name: str, destination: str, start_date: str, end_date: str) -> dict:
    """Create a new trip for a user. Dates in YYYY-MM-DD format."""
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("SELECT user_id FROM users WHERE name = %s", (user_name,))
    row = cursor.fetchone()
    user_id = row[0] if row else None
    if user_id is None:
        cursor.execute("INSERT INTO users (name) VALUES (%s) RETURNING user_id", (user_name,))
        user_id = cursor.fetchone()[0]
    cursor.execute(
        "INSERT INTO trips (user_id, destination, start_date, end_date) VALUES (%s, %s, %s, %s) RETURNING trip_id",
        (user_id, destination, start_date, end_date)
    )
    trip_id = cursor.fetchone()[0]
    conn.commit()
    cursor.close()
    conn.close()
    return {"trip_id": trip_id, "user_id": user_id, "destination": destination}

def add_itinerary_item(trip_id: int, day_number: int, activity: str, notes: str = "") -> dict:
    """Add an activity to a trip's itinerary for a specific day."""
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO itinerary_items (trip_id, day_number, activity, notes) VALUES (%s, %s, %s, %s) RETURNING item_id",
        (trip_id, day_number, activity, notes)
    )
    item_id = cursor.fetchone()[0]
    conn.commit()
    cursor.close()
    conn.close()
    return {"item_id": item_id, "trip_id": trip_id, "day_number": day_number, "activity": activity}

def add_packing_item(trip_id: int, item_name: str) -> dict:
    """Add an item to a trip's packing list."""
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO packing_items (trip_id, item_name) VALUES (%s, %s) RETURNING item_id",
        (trip_id, item_name)
    )
    item_id = cursor.fetchone()[0]
    conn.commit()
    cursor.close()
    conn.close()
    return {"item_id": item_id, "trip_id": trip_id, "item_name": item_name}

print("All 5 tools defined.")

In [0]:
import json
from datetime import date

DATABRICKS_HOST = w.config.host
DATABRICKS_TOKEN = w.config.token

def call_llm(messages, tools):
    headers = w.config.authenticate()
    headers["Content-Type"] = "application/json"

    resp = requests.post(
        f"{w.config.host}/serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations",
        headers=headers,
        json={
            "messages": messages,
            "tools": tools,
            "tool_choice": "auto",
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]

def run_agent(user_message: str):
    today_str = date.today().isoformat()
    messages = [
        {"role": "system", "content": f"Today's date is {today_str}. Use this as the reference point for any relative dates the user mentions."},
        {"role": "user", "content": user_message}
    ]

    for _ in range(5):
        choice = call_llm(messages, tool_schemas)

        if not choice.get("tool_calls"):
            return choice.get("content")

        messages.append({
            "role": "assistant",
            "content": choice.get("content") or "",
            "tool_calls": choice["tool_calls"],
        })

        for call in choice["tool_calls"]:
            fn_name = call["function"]["name"]
            args = json.loads(call["function"]["arguments"])
            print(f"[Agent calling tool: {fn_name}({args})]")
            result = tool_functions[fn_name](**args)
            messages.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "content": str(result)
            })

    return "Reached max tool-call iterations."

print(run_agent("I want to plan a trip to Kyoto focused on historic temples. What's the weather like there right now?"))

In [0]:
print(run_agent("Create a trip to Kyoto for Alex, October 1st to October 5th, and add visiting Kiyomizu-dera as a day 1 activity."))